# PyTorch 기본 신경망 (Basic Neural Networks)

이 노트북에서는 PyTorch의 기본 개념과 간단한 신경망을 구현합니다.

## 학습 목표
1. PyTorch Tensor 연산 이해
2. Autograd (자동 미분) 사용법
3. 다층 퍼셉트론(MLP) 구현
4. MNIST 손글씨 숫자 분류
5. 모델 저장 및 로드

In [ ]:
# 필요한 라이브러리 임포트
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. PyTorch Tensor 기초

Tensor는 PyTorch의 기본 데이터 구조입니다. NumPy 배열과 유사하지만 GPU에서 실행할 수 있습니다.

In [ ]:
# Tensor 생성 방법
x = torch.tensor([1, 2, 3, 4, 5])
print("1D Tensor:", x)

y = torch.zeros(3, 4)  # 3x4 영행렬
print("\nZero Tensor:\n", y)

z = torch.randn(2, 3)  # 정규분포에서 샘플링
print("\nRandom Tensor:\n", z)

# Tensor 연산
a = torch.tensor([[1, 2], [3, 4]])
b = torch.tensor([[5, 6], [7, 8]])
print("\nElement-wise addition:\n", a + b)
print("\nMatrix multiplication:\n", torch.matmul(a, b))

## 2. Autograd: 자동 미분

PyTorch의 자동 미분 기능은 역전파를 자동으로 계산합니다.

In [ ]:
# requires_grad=True로 gradient 추적
x = torch.tensor([2.0, 3.0, 4.0], requires_grad=True)
print("x:", x)

# 연산 수행
y = x ** 2 + 2 * x + 1
print("y = x^2 + 2x + 1:", y)

# 역전파
z = y.sum()
z.backward()

# Gradient 확인 (dy/dx = 2x + 2)
print("\nGradient (dy/dx):", x.grad)
print("Expected: 2*x + 2 =", 2 * x.detach() + 2)

## 3. 데이터 로드: MNIST 데이터셋

손글씨 숫자 (0-9) 이미지를 분류하는 작업입니다.

In [ ]:
# 데이터 전처리: 정규화
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))  # MNIST 평균과 표준편차
])

# 데이터셋 다운로드
train_dataset = torchvision.datasets.MNIST(
    root='../data',
    train=True,
    download=True,
    transform=transform
)

test_dataset = torchvision.datasets.MNIST(
    root='../data',
    train=False,
    download=True,
    transform=transform
)

# DataLoader 생성
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

In [ ]:
# 데이터 시각화
examples = iter(train_loader)
example_data, example_targets = next(examples)

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(example_data[i].squeeze(), cmap='gray')
    ax.set_title(f"Label: {example_targets[i]}")
    ax.axis('off')
plt.tight_layout()
plt.show()

## 4. 다층 퍼셉트론(MLP) 모델 정의

3개의 완전 연결 레이어로 구성된 간단한 신경망을 만듭니다.

In [ ]:
class SimpleMLP(nn.Module):
    def __init__(self, input_size=784, hidden_size=128, num_classes=10):
        super(SimpleMLP, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(hidden_size, num_classes)
    
    def forward(self, x):
        # Flatten: (batch_size, 1, 28, 28) -> (batch_size, 784)
        x = x.view(x.size(0), -1)
        x = self.relu1(self.fc1(x))
        x = self.relu2(self.fc2(x))
        x = self.fc3(x)
        return x

# 모델 생성
model = SimpleMLP().to(device)
print(model)

# 파라미터 수 계산
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")

## 5. 손실 함수와 옵티마이저 정의

In [ ]:
# 손실 함수: Cross Entropy Loss
criterion = nn.CrossEntropyLoss()

# 옵티마이저: Adam
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 학습률 스케줄러 (선택사항)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

## 6. 학습 함수 정의

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(dataloader, desc='Training')
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Statistics
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        pbar.set_postfix({'loss': loss.item(), 'acc': 100 * correct / total})
    
    epoch_loss = running_loss / len(dataloader)
    epoch_acc = 100 * correct / total
    return epoch_loss, epoch_acc

## 7. 평가 함수 정의

In [ ]:
def evaluate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc='Evaluating'):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / len(dataloader)
    epoch_acc = 100 * correct / total
    return epoch_loss, epoch_acc

## 8. 모델 학습

In [ ]:
num_epochs = 10
train_losses, train_accs = [], []
test_losses, test_accs = [], []

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    
    # 학습
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    
    # 평가
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)
    test_losses.append(test_loss)
    test_accs.append(test_acc)
    
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
    print(f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.2f}%")
    
    # 학습률 조정
    scheduler.step()

print("\nTraining completed!")

## 9. 학습 결과 시각화

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss 그래프
ax1.plot(train_losses, label='Train Loss', marker='o')
ax1.plot(test_losses, label='Test Loss', marker='s')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Loss over Epochs')
ax1.legend()
ax1.grid(True)

# Accuracy 그래프
ax2.plot(train_accs, label='Train Accuracy', marker='o')
ax2.plot(test_accs, label='Test Accuracy', marker='s')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Accuracy over Epochs')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

## 10. 예측 결과 시각화

In [ ]:
# 테스트 샘플로 예측
model.eval()
examples = iter(test_loader)
example_data, example_targets = next(examples)
example_data = example_data.to(device)

with torch.no_grad():
    outputs = model(example_data)
    _, predictions = torch.max(outputs, 1)

# 결과 시각화
fig, axes = plt.subplots(3, 5, figsize=(15, 9))
for i, ax in enumerate(axes.flat):
    ax.imshow(example_data[i].cpu().squeeze(), cmap='gray')
    pred = predictions[i].item()
    true = example_targets[i].item()
    color = 'green' if pred == true else 'red'
    ax.set_title(f"Pred: {pred}, True: {true}", color=color)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 11. 모델 저장 및 로드

In [ ]:
# 모델 저장
torch.save({
    'epoch': num_epochs,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'train_loss': train_losses[-1],
    'test_loss': test_losses[-1],
}, '../checkpoints/mnist_mlp.pth')

print("Model saved to '../checkpoints/mnist_mlp.pth'")

# 모델 로드
checkpoint = torch.load('../checkpoints/mnist_mlp.pth')
model_loaded = SimpleMLP().to(device)
model_loaded.load_state_dict(checkpoint['model_state_dict'])
print(f"Model loaded from checkpoint (epoch {checkpoint['epoch']})")

## 12. 혼동 행렬 (Confusion Matrix)

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

# 전체 테스트 셋에 대한 예측
all_preds = []
all_labels = []

model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, predictions = torch.max(outputs, 1)
        all_preds.extend(predictions.cpu().numpy())
        all_labels.extend(labels.numpy())

# 혼동 행렬 계산
cm = confusion_matrix(all_labels, all_preds)

# 시각화
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

## 연습 문제

1. **드롭아웃 추가**: 모델에 `nn.Dropout(0.5)`를 추가하고 성능 변화를 관찰하세요.
2. **배치 정규화**: `nn.BatchNorm1d`를 추가해보세요.
3. **다른 활성화 함수**: ReLU 대신 LeakyReLU, ELU 등을 시도해보세요.
4. **하이퍼파라미터 튜닝**: learning rate, batch size, hidden size를 변경해보세요.
5. **Early Stopping**: 검증 손실이 증가하면 학습을 중단하는 로직을 추가하세요.

## 다음 단계

- [02_cnn.ipynb](./02_cnn.ipynb): CNN으로 이미지 분류 성능 향상
- [03_rnn_lstm.ipynb](./03_rnn_lstm.ipynb): 시퀀스 데이터 처리
- [04_transformer.ipynb](./04_transformer.ipynb): Transformer 아키텍처 구현